**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/allen-2026"], check=True)
    os.chdir("/content/encore/entries/allen-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Allen, Yan, Kuan et al. (2026) from Dryad rjdfn2zqz

**Paper.** Allen, Yan, Kuan et al. (2026). Neurometabolic rewiring in squid (*Sepioteuthis lessoniana*)
optic lobes drives behavioral plasticity and visual integration under environmental acidification.
*Communications Biology* 9:229. [doi:10.1038/s42003-025-09506-6](https://doi.org/10.1038/s42003-025-09506-6).
The claims file (`claims/allen-2026.yaml`) was written from the version of record and its
supplement (Supplementary Figures 1 to 5, Supplementary Tables 1 to 5).

**Dataset.** Dryad [doi:10.5061/dryad.rjdfn2zqz](https://doi.org/10.5061/dryad.rjdfn2zqz), version id
439366 (API `/api/v2/versions/439366`, version number 6, published 2026-04-29). The paper cites the
dataset's DOI, which resolves to the latest version, and no version. 439366 is the latest and the
only version the API lists, so it is the one pinned. The files are read from the directory where
the workflow stages them, after checking each against the SHA-256 of Dryad's file list (below).

**What the dataset holds.** One CSV per measure, a value per squid with its group: routine
metabolic rate, ammonia excretion, hemolymph pH, PCO2 and [HCO3-] (with the calculation of PCO2 and
[HCO3-] in `Carbonate_calibration.csv`), normalized distance and velocity, time in the upper, left
and front zones, prey capture, hunt duration and strike attempts, critical flicker fusion per
squid, and the ERG ratio as group means per stimulus frequency. `squid.zip` holds the authors'
shell and R scripts, the ExN50 statistics of the assembly, the RSEM count matrix (902,413
transcripts by 16 samples) and their R workspace `Squid.RData`, which holds the DESeq2, sva and
GSEA objects of the transcriptome analysis and the per-figure tables of Figures 1 to 3 and
Supplementary Figures 2 and 3. Not in the dataset: the 3-D tracks, ERG traces, per-squid ERG
spectra, seawater chemistry, body mass and mantle length, and the RNA-seq reads (in SRA).

**Outside data.** One number: the count of sequences of the paper's public assembly, read from its
NCBI TSA master record (E-utilities esummary) as a variant of the transcript count of
rnaseq-assembly. The transcriptomic claims are marked `data: elsewhere` (SRA
PRJNA1204452, TSA GLBY01000000) in the claims file; they are tested here on the count matrix and the
authors' DESeq2, sva and GSEA objects that this dataset holds, as the reads would have to be
re-assembled with Trinity (100 GB of memory in the authors' script) to start from SRA.

| Claim | Tested here |
|---|---|
| fig1b-metabolic-rate, fig1c-ammonia-excretion | yes, deposited rates |
| fig1d-blood-ph, fig1e-blood-pco2, fig1f-blood-bicarbonate | yes, deposited values |
| methods-body-size | blocked, data missing (no mass or mantle length) |
| stable5-seawater-chemistry | blocked, data missing (no seawater chemistry) |
| sfig2-distance-unchanged, fig2c-velocity-unchanged | yes, deposited per-squid values |
| fig2d-acute-upper-zone, fig2e-left-right-unchanged, fig2f-front-back-unchanged | yes, deposited zone percentages |
| stable3-acute-shallower | blocked, data missing (no depth or tracks) |
| fig2g-attack-willingness | yes, counts of the workspace's Figure 2G table (the per-squid file lacks 11 chronic-OA squid) |
| fig2h-hunt-duration, fig2i-strike-attempts | yes |
| sfig3-prey-velocity-unchanged | in part, table `S3` of the R workspace (the CO2 10 to 20 min column is a copy) |
| fig3c-ratio-declines | yes, deposited group means per frequency |
| sfig4a-amplitude-attenuation | blocked, data missing (no ERG traces) |
| fig3d-cff-unchanged, fig3d-cff-variance | yes, deposited CFF per squid |
| rnaseq-assembly | in part, the mapping rate is not deposited |
| fig4a-pca, sfig5b-acute-clusters-with-control | yes, the authors' batch-corrected VST matrix |
| fig4b to fig4fgh, chronic-nachr-subunits-down, optic-lobe-no-ion-regulation | yes, the authors' saved GSEA and DESeq2 results, not re-run |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

import squidoa as sq

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
warnings.filterwarnings("ignore", category=FutureWarning)
started = time.time()
print(f"Dryad {sq.DRYAD_ID}, version id {sq.DRYAD_VERSION}")
print("SHA-256 against Dryad's file list:", sq.check_files())

## Data check

In [ ]:
print("Files in squid.zip (bytes):")
for name, size in sq.zip_names():
    print(f"  {size:>11,}  {name}")
csvs = {n: sq.read_csv(n) for n in sorted(sq.SHA256) if n.endswith(".csv")}
for name, df in csvs.items():
    print(f"\n{name}: {df.shape[0]} rows; columns {list(df.columns)}")
    if "Group" in df.columns:
        print("  rows per group:", df.Group.value_counts().reindex(sq.GROUPS + sorted(set(df.Group) - set(sq.GROUPS))).dropna().astype(int).to_dict())
    num = df.select_dtypes("number")
    if len(num.columns):
        print(num.describe().loc[["count", "min", "max"]].round(4).to_string())

In [ ]:
rmr_file = csvs["Squid_RMR.csv"]
print("Squid_RMR.csv squid per group:", rmr_file.Group.value_counts().reindex(sq.GROUPS).to_dict(), "total", len(rmr_file))
print("Supplementary Table 1: F(2,26), total df 28, so 29 squid")

Group sizes against the paper. Routine metabolic rate: 11, 9 and 6 squid (26), where the Fig. 1
legend gives n = 8 to 12 and Supplementary Table 1's ANOVA has 26 residual degrees of freedom, that
is 29 squid. Ammonia: 5, 9, 7 (21, as the ANOVA's df). Hemolymph: 7, 6, 9, as Supplementary Table 2.
Distance and velocity: 15, 8, 11, as Supplementary Table 3's df. Zone durations: 15, 9, 11, one
acute-OA squid more than the ANOVA's df (31 residual df, 34 squid). Prey capture: 46, 36, 22, as the
logistic regression's n. CFF: 3, 3, 6 (Fig. 3D, n = 3 to 6). The ERG file holds group means and
SDs per stimulus frequency, not values per squid. The README calls the hemolymph PCO2 column
"(atm)"; its header and values are in µatm (785.9 to 3861.9).

In [ ]:
t0 = time.time()
ws = sq.read_rdata({"behavior_and_physiology_data", "df", "svseq", "vst", "vstsva.adj", "vstsva.adj.limma",
                    "resNormal", "gse.Normal", "DB"})
print(f"read Squid.RData objects in {time.time() - t0:.0f} s")
bp = ws["behavior_and_physiology_data"]
fig_tables = {n: sq.df_from(v) for n, v in zip(bp.names(), bp.value)}
for n, d in fig_tables.items():
    gcol = "group" if "group" in d.columns else ("Group" if "Group" in d.columns else None)
    counts = d.dropna(subset=[c for c in d.columns if c not in ("group", "Group")][:1]).groupby(gcol).size().to_dict() if gcol else None
    print(f"{n}: {d.shape}, columns {list(d.columns)}, non-missing per group {counts}")

In [ ]:
# The per-figure tables of the R workspace against the CSV files: same values per group?
pairs = {"F1B": ("Squid_RMR.csv", 0), "F1C": ("Squid_Ammonia.csv", 0), "F1D": ("Squid_Hemolymph.csv", 0),
         "F1E": ("Squid_Hemolymph.csv", 1), "F1F": ("Squid_Hemolymph.csv", 2), "F2C": ("Squid_normalized_total_distance.csv", 0),
         "S2": ("Squid_normalized_total_distance.csv", 0), "F2D": ("Squid_Behavioral_assessment.csv", 0),
         "F2E": ("Squid_Behavioral_assessment.csv", 1), "F2F": ("Squid_Behavioral_assessment.csv", 2),
         "F2H": ("Squid_Prey_Capture.csv", 2), "F2I": ("Squid_Prey_Capture.csv", 3), "F3D": ("Squid_CFF.csv", 0)}
for fig, (fname, ci) in pairs.items():
    d = fig_tables[fig].dropna()
    d["group"] = d["group"].replace({"Acute": "Acute-OA", "Chronic": "Chronic-OA"})
    c = csvs[fname]
    same = all(np.allclose(np.sort(d.loc[d.group == g, "value"]), np.sort(c.loc[c.Group == g, c.columns[ci]].dropna()))
               if (d.group == g).sum() == c.loc[c.Group == g, c.columns[ci]].notna().sum() else False for g in sq.GROUPS)
    print(f"{fig} against {fname} column {c.columns[ci]!r}: same values per group: {same}")
v = csvs["Squid_normalized_velocities.csv"]; dist = csvs["Squid_normalized_total_distance.csv"]
ratio = v.iloc[:, 0].to_numpy() / dist.iloc[:, 0].to_numpy()
print("\nvelocity / distance per squid (s^-1): min", ratio.min().round(8), "max", ratio.max().round(8), "-> 1/600 =", round(1 / 600, 8))
print("Groups in the same order in both files:", bool((v.Group.to_numpy() == dist.Group.to_numpy()).all()))

The R workspace's figure tables hold the same values as the CSV files (table `F2C`, which the
script plots as Figure 2C, holds the distances, not the velocities). Velocity is distance divided
by 600 s for every squid, so the two files carry one measure. The workspace's `S3` table is the
only copy of the prey shrimp velocities (Supplementary Fig. 3).

In [ ]:
# Every top-level object of Squid.RData (name, R type, serialized bytes), and the README inside squid.zip
for name, typ, size in sq.top_level_names():
    print(f"  {name:32s} {typ:12s} {size:>12,}")
print("\nsquid.zip README.md:\n" + sq.zip_text("README.md"))

The workspace holds the transcriptome objects (count matrices, DESeq2 fits `dds` and `ddssva`, VST
matrices, sva, DESeq2 and GSEA results, the annotation table `DB` and clusterProfiler's caches) and
one list of figure tables, `behavior_and_physiology_data`, whose tables are printed above. No object
holds body mass, mantle length, seawater chemistry, 3-D tracks or depth, or ERG traces, spectra or
per-squid ratios; the zip's README lists only the scripts, ExN50 statistics, count matrix and
workspace.

In [ ]:
cc = csvs["Carbonate_calibration.csv"]
alpha, pka = 0.0312, 6.10   # as printed in each row of Carbonate_calibration.csv
eq1 = cc["TCO2 (mM)"] / ((1 + 10 ** (cc["pH"] - pka)) * alpha) * 1e3   # Eq. 1 as the file's formula column writes it
print(cc[["sample_name", "PCO2 Calculation with Values", "PCO₂ (μatm)"]].head(3).to_string(index=False))
print("\nThe formula as written, evaluated (first rows):", eq1.head(3).round(1).tolist())
ratio = eq1 / cc["PCO₂ (μatm)"]
print(f"Formula value / PCO2 printed beside it: min {ratio.min():.4f}, max {ratio.max():.4f}")
hem = csvs["Squid_Hemolymph.csv"]
print("Squid_Hemolymph PCO2 equals the printed PCO2 of Carbonate_calibration (max |diff|):",
      round(float(np.abs(hem.iloc[:, 1].to_numpy() - cc["PCO₂ (μatm)"].to_numpy()).max()), 3))
hco3_dep = cc["TCO2 (mM)"] - alpha * cc["PCO₂ (μatm)"] / 1e3
hco3_eq = cc["TCO2 (mM)"] - alpha * eq1 / 1e3
print("Eq. 2 with the printed PCO2 reproduces the deposited [HCO3-] (max |diff|):", round(float(np.abs(hco3_dep.to_numpy() - hem.iloc[:, 2].to_numpy()).max()), 4))
print("Eq. 2 with the formula's PCO2 differs from the deposited [HCO3-] by up to", round(float(np.abs(hco3_eq.to_numpy() - hem.iloc[:, 2].to_numpy()).max()), 3), "mmol/l")
print("pH identical in both files:", bool(np.allclose(cc.pH.to_numpy(), hem.iloc[:, 0].to_numpy())))
co2 = cc["TCO2 (mM)"] / (1 + 10 ** (cc["pH"] - pka))   # dissolved CO2 (mmol/l) from the measured TCO2 and pH
print("\nDissolved CO2 from the measured TCO2 and pH, mmol/l (first rows):", co2.head(3).round(3).tolist())
print("alphaCO2 that would give the printed PCO2 (mmol l-1 per matm): min", round(float((co2 / (cc["PCO₂ (μatm)"] / 1e3)).min()), 4),
      "max", round(float((co2 / (cc["PCO₂ (μatm)"] / 1e3)).max()), 4))
print("Group means of [HCO3-], deposited and with the formula's PCO2 (mmol/l):")
print(pd.DataFrame({"deposited": hem.iloc[:, 2].groupby(hem.Group.to_numpy()).mean(),
                    "formula PCO2": hco3_eq.groupby(hem.Group.to_numpy()).mean()}).reindex(sq.GROUPS).round(3).to_string())
print("Drop in the group means of [HCO3-] with the formula's PCO2 (mmol/l):",
      (hem.iloc[:, 2].groupby(hem.Group.to_numpy()).mean() - hco3_eq.groupby(hem.Group.to_numpy()).mean()).reindex(sq.GROUPS).round(3).to_dict())
# The tests of fig1e-blood-pco2 on the formula's PCO2, to compare with those on the deposited values (claim cell below)
for label, values in [("deposited PCO2 (Squid_Hemolymph.csv)", hem.iloc[:, 1]), ("formula's PCO2", eq1)]:
    g = sq.by_group(hem.assign(pco2_test=values.to_numpy()), "pco2_test")
    w = sq.welch_anova(list(g.values()))
    m = {k: v.mean() for k, v in g.items()}
    print(f"\n{label}: fold change acute/control {m['Acute-OA'] / m['Control']:.4f}, chronic/control {m['Chronic-OA'] / m['Control']:.4f}; "
          f"Welch's ANOVA W = {w['W']:.3f}, df 2, {w['df2']:.3f}, p = {w['p']:.3g}")
    print(sq.dunnett_t3(g)[["comparison", "p_adj"]].round(4).to_string(index=False))

The deposited hemolymph PCO2 is a tenth of what the formula printed beside it gives: each row
writes Eq. 1 with αCO2 = 0.0312 and pKa1 = 6.10 and "×10³", which evaluates to 10 times the PCO2
printed in the same row (and in `Squid_Hemolymph.csv` and the paper). The deposited [HCO3-] follows
from Eq. 2 with the printed PCO2. The formula is the internally consistent one: the README gives
αCO2 as the value for 32 ppt and 25 °C, and CO2 solubility in seawater there is about 0.03 mmol l-1
per matm (µmol l-1 per µatm), so "×10³" turns mmol l-1 over αCO2 into µatm. The measured TCO2 and pH
give a dissolved CO2 of 0.37 mmol/l in the first row, which matches the printed PCO2 only with an
αCO2 near 0.31, ten times the stated value. The deposited PCO2, and the paper's, are therefore
consistent with the measured TCO2 and pH only if the stated αCO2 is wrong by a factor of 10; the
data do not show which value the authors meant. With the formula's PCO2 the group means of
[HCO3-] fall by 0.30, 0.45 and 0.78 mmol/l (control, acute-OA, chronic-OA). The criteria test the
values as deposited. The ratio between the two PCO2 scales varies from 9.997 to 10.049 between squid,
so the fold changes and tests of PCO2 on the formula's scale, printed above, are close to those on
the deposited values but not identical.

## fig1b-metabolic-rate

> Routine metabolic rate was 16.9 (control), 18.9 (acute-OA) and 23.9 (chronic-OA) umol O2 per g
> fresh mass per h; chronic-OA squid had a 41% higher rate than controls, and acute-OA did not
> differ from either.

The deposit gives the rate per squid; untransformed one-way ANOVA and Tukey's test (Supplementary
Table 1).

In [ ]:
rmr = sq.by_group(csvs["Squid_RMR.csv"], csvs["Squid_RMR.csv"].columns[0])
print(sq.describe(rmr).round(3).to_string())
res = sq.oneway(rmr)
print(f"\nANOVA F({res['df1']},{res['df2']}) = {res['F']:.3f}, p = {res['p']:.6f}; SS treatment {res['SS_treat']:.1f}, residual {res['SS_resid']:.1f}, eta^2 = {res['eta2']:.3f}")
print("Paper: F(2,26) = 4.473, p = 0.021407; SS treatment 235.4, residual 684.2, total 919.6")
print("Tukey adjusted p:", {k: round(v, 4) for k, v in res["tukey"].items()})
inc = 100 * (rmr["Chronic-OA"].mean() / rmr["Control"].mean() - 1)
print(f"Chronic-OA over control: +{inc:.1f}%")

**Result: reproduced.** The means are 16.88, 18.93 and 23.88 µmol O2 g-1 h-1, chronic-OA 41.5%
above control; the ANOVA gives p = 0.0375 and Tukey's test separates control from chronic-OA (p =
0.030) but neither from acute-OA. The deposit holds 26 squid, three fewer than the paper's ANOVA
(F(2,23) = 3.80 here against F(2,26) = 4.473).

## fig1c-ammonia-excretion

> Ammonia excretion was similar in control (3.3) and acute-OA (2.9 umol per g per h) squid and
> lower in chronic-OA squid (1.6), about 52% and 45% below control and acute-OA.

One-way ANOVA on log10 rates and Tukey's test (Supplementary Table 1).

In [ ]:
amm = sq.by_group(csvs["Squid_Ammonia.csv"], csvs["Squid_Ammonia.csv"].columns[0])
print(sq.describe(amm).round(3).to_string())
res = sq.oneway({g: np.log10(v) for g, v in amm.items()})
print(f"\nANOVA on log10: F({res['df1']},{res['df2']}) = {res['F']:.3f}, p = {res['p']:.6f}; SS treatment {res['SS_treat']:.4f}, residual {res['SS_resid']:.4f}; eta^2 = {res['eta2']:.3f}")
print("Paper: F(2,18) = 6.382, p = 0.008038; SS 0.3563 and 0.5025")
print("Tukey adjusted p (log10):", {k: round(v, 4) for k, v in res["tukey"].items()})
raw = sq.oneway(amm)
print(f"Variant, untransformed: F = {raw['F']:.3f}, p = {raw['p']:.4f}; Tukey", {k: round(v, 4) for k, v in raw["tukey"].items()})
m = {g: v.mean() for g, v in amm.items()}
print(f"Reduction of chronic-OA against control {100 * (1 - m['Chronic-OA'] / m['Control']):.1f}%, against acute-OA {100 * (1 - m['Chronic-OA'] / m['Acute-OA']):.1f}%")

**Result: reproduced.** Means 3.34, 2.86 and 1.61 µmol g-1 h-1, reductions of 51.8% and 43.6%. The
ANOVA on log rates gives F(2,18) = 6.382, p = 0.008038, the paper's values; Tukey separates chronic-OA
from control (p = 0.009) and acute-OA (p = 0.038), not control from acute-OA (p = 0.52).

## fig1d-blood-ph, fig1e-blood-pco2, fig1f-blood-bicarbonate

> Blood pH was higher in chronic-OA squid (7.17) than in controls (7.09); acute-OA squid were
> intermediate (7.15) and differed from neither.
>
> Blood PCO2 rose 1.5-fold under acute-OA (1583.8 uatm) and 2.6-fold under chronic-OA (2778.7 uatm)
> from 1057.4 uatm in controls, and all three groups differ.
>
> Blood [HCO3-] rose 1.7-fold under acute-OA (5.90 mmol/l) and 3.1-fold under chronic-OA (11.08
> mmol/l) from 3.56 mmol/l in controls, and all three groups differ.

PCO2 and [HCO3-] as deposited. Welch's ANOVA and Dunnett's T3 (Methods, Statistics; Supplementary
Table 2), implemented in `squidoa.welch_anova` and `squidoa.dunnett_t3`.

In [ ]:
blood = {}
for col, key in zip(hem.columns[:3], ["pH", "PCO2", "HCO3"]):
    g = sq.by_group(hem, col)
    blood[key] = g
    w = sq.welch_anova(list(g.values()))
    print(f"--- {col}")
    print(sq.describe(g).round(3).to_string())
    print(f"Welch's ANOVA W = {w['W']:.3f}, df 2, {w['df2']:.3f}, p = {w['p']:.3g}")
    print(sq.dunnett_t3(g).round(4).to_string(index=False))
    if key != "pH":
        m = {k: v.mean() for k, v in g.items()}
        print(f"fold change acute/control {m['Acute-OA'] / m['Control']:.2f}, chronic/control {m['Chronic-OA'] / m['Control']:.2f}")
print("\nPaper (Supplementary Table 2): pH W = 5.983, df 2, 9.196, p = 0.00965; PCO2 W = 27.653, df 2, 9.379; HCO3 W = 39.545, df 2, 12.59")
print("The paper's pH line: W = 5.983 on (2, 9.196) df gives p =", round(float(stats.f.sf(5.983, 2, 9.196)), 4), "(printed 0.00965)")

**Result: fig1d-blood-ph reproduced.** Means 7.092, 7.145 and 7.175 (n = 7, 6, 9); Welch's ANOVA p
= 0.0185 (W = 6.347 against the paper's 5.983, with the same df 2, 9.196); Dunnett's T3 separates
control from chronic-OA (p = 0.017) and neither pair with acute-OA (p = 0.46 and 0.86). The paper's
pair p-values (0.015, 0.391, 0.754) and its narrower intervals do not follow from these values with
the studentized maximum modulus critical value used here (2.83 at 10 df); its intervals imply a
critical value near 1.96. The paper's own line is also inconsistent: W = 5.983 on (2, 9.196) df
gives p = 0.022, not the printed 0.00965. The verdicts do not depend on these differences.

**Result: fig1e-blood-pco2 reproduced.** Means 1057.4, 1583.8 and 2778.7 µatm, fold changes 1.50
and 2.63; Welch's p = 2.2e-05; all three pairs differ (p = 0.014, below 0.0001 and 0.0008). This
Welch's W (29.815, df 2, 12.04) differs from the paper's (27.653, df 2, 9.379) on the same values.
The values are a tenth of what the deposit's own Eq. 1 gives (data check); the fold changes and
tests do not depend on that factor.

**Result: fig1f-blood-bicarbonate reproduced.** Means 3.56, 5.90 and 11.08 mmol/l, fold changes
1.65 and 3.11; Welch's p = 2.5e-06; all pairs differ (p ≤ 0.0008).

## methods-body-size, stable5-seawater-chemistry, stable3-acute-shallower

Blocked, data missing. No file holds fresh mass or mantle length, seawater pH or PCO2 (the
carbonate calculation file covers hemolymph only), or the depth of the squid (only the percentage
of time in each zone is deposited; the 3-D tracks are not).

## sfig2-distance-unchanged

> Total distance travelled during the 10 min exploration did not differ between groups (132.9,
> 124.4 and 147.3 mantle lengths for control, acute-OA and chronic-OA).

The deposit gives the normalized distance per squid (the 3-D tracks are not deposited, so Eq. 3 and
the handling of untracked samples cannot be applied). Kruskal-Wallis at p < 0.05 (Supplementary
Table 3).

In [ ]:
dist_g = sq.by_group(dist, dist.columns[0])
print(sq.describe(dist_g).round(2).to_string())
kw = stats.kruskal(*dist_g.values())
n = sum(len(v) for v in dist_g.values())
print(f"\nKruskal-Wallis H = {kw.statistic:.4f}, p = {kw.pvalue:.6f}; epsilon^2 = {kw.statistic / (n - 1):.4f}")
print("Paper: KW = 0.4124, p = 0.813689")

**Result: reproduced.** Means 132.9, 124.4 and 147.4 mantle lengths (15, 8, 11 squid);
Kruskal-Wallis H = 0.4124, p = 0.8137, the paper's values; effect size ε² = 0.013.

## fig2c-velocity-unchanged

> Mean swimming velocity during the 10 min exploration did not differ between groups (0.2, 0.21
> and 0.25 mantle lengths per s for control, acute-OA and chronic-OA).

Velocity per squid as deposited, which is the distance over 600 s (data check). Kruskal-Wallis at
p < 0.05. The velocity file has 8 acute-OA squid, so no 9th squid is shown.

In [ ]:
vel_g = sq.by_group(v, v.columns[0])
print(sq.describe(vel_g).round(4).to_string())
kw = stats.kruskal(*vel_g.values())
print(f"\nKruskal-Wallis H = {kw.statistic:.4f}, p = {kw.pvalue:.6f}; epsilon^2 = {kw.statistic / (sum(len(x) for x in vel_g.values()) - 1):.4f}")
print("Paper: KW = 0.4124, p = 0.813689")

**Result: reproduced.** Means 0.2215, 0.2074 and 0.2456 mantle lengths/s; the control mean rounds to
0.2 and lies within 0.15 to 0.25. Kruskal-Wallis H = 0.4124, p = 0.8137.

## fig2d-acute-upper-zone, fig2e-left-right-unchanged, fig2f-front-back-unchanged

> Acute-OA squid spent most of the exploration in the upper half of the arena (86.0%), more than
> control (30.0%) and chronic-OA (14.7%) squid, which did not differ from each other.
>
> Acidification did not change the preference for the left or right side of the arena (55.8%,
> 54.1% and 57.1% of the trial on the left for control, acute-OA and chronic-OA).
>
> Acidification did not change the preference for the front or back of the arena (58.3%, 80.0%
> and 66.1% of the trial near the front wall for control, acute-OA and chronic-OA).

The deposit gives the percentage per squid; the zone boundaries and the denominator (readings (a)
and (b) of the criteria) were applied before deposit and cannot be varied here. The file has 9
acute-OA squid where the paper's ANOVAs have 8 (31 residual df). Each acute-OA squid is left out in
turn, and the one whose exclusion gives the paper's sums of squares (Supplementary Table 3) is
taken as the paper's set; all 9 are shown as a variant. Untransformed one-way ANOVA and Tukey's test;
for left and front, log10 as a variant (zeros left out).

In [ ]:
beh = csvs["Squid_Behavioral_assessment.csv"]
paper_ss = {"Duration in Upper zone (%)": 25398, "Duration in Left zone (%)": 42.93, "Duration in Front zone (%)": 2437}
acute_rows = beh.index[beh.Group == "Acute-OA"]
rows = []
for drop in acute_rows:
    r = {"left_out_row": drop}
    for col in paper_ss:
        g = sq.by_group(beh.drop(index=drop), col)
        r[col.split(" ")[2] + "_SS"] = round(sq.oneway(g)["SS_treat"], 1)
    rows.append(r)
loo = pd.DataFrame(rows)
print("Treatment sums of squares with one acute-OA squid left out (paper: upper 25398, left 42.93, front 2437):")
print(loo.to_string(index=False))
dist_to_paper = sum((loo[c.split(" ")[2] + "_SS"] - s) ** 2 / s ** 2 for c, s in paper_ss.items())
paper_drop = int(loo.left_out_row[dist_to_paper.idxmin()])
print("\nLeft out to match the paper: row", paper_drop, beh.loc[paper_drop].to_dict())
# The workspace's figure tables F2D (upper zone) and F2C (distance) both hold 15 rows per group. That a row of
# one is the same squid as the same row of the other is assumed: no identifier links them. Under that
# assumption the acute-OA row with a zone value and no distance is the squid the distance file lacks.
f2 = fig_tables["F2D"].assign(distance=fig_tables["F2C"]["value"].to_numpy())
f2 = f2[f2.group == "Acute-OA"]
print("\nAcute-OA rows of F2D and F2C side by side:")
print(f2.rename(columns={"value": "upper %"}).to_string())
print("Upper-zone value with no distance:", f2.loc[f2.distance.isna() & f2["value"].notna(), "value"].round(4).tolist(),
      "; the squid left out above:", round(beh.loc[paper_drop, "Duration in Upper zone (%)"], 4))

In [ ]:
zone_res = {}
for col, claim in [("Duration in Upper zone (%)", "fig2d"), ("Duration in Left zone (%)", "fig2e"), ("Duration in Front zone (%)", "fig2f")]:
    for label, data in [("paper set (8 acute-OA)", beh.drop(index=paper_drop)), ("all 9 acute-OA", beh)]:
        g = sq.by_group(data, col)
        r = sq.oneway(g)
        zone_res[(claim, label)] = r
        d = sq.describe(g)
        print(f"--- {col}, {label}")
        print("  mean:", d["mean"].round(1).to_dict(), " SD:", d["sd"].round(1).to_dict(), " n:", d["n"].astype(int).to_dict())
        print(f"  ANOVA F({r['df1']},{r['df2']}) = {r['F']:.4f}, p = {r['p']:.6f}, SS treatment {r['SS_treat']:.1f}, residual {r['SS_resid']:.1f}, eta^2 = {r['eta2']:.3f}")
        print("  Tukey adjusted p:", {k: round(x, 4) for k, x in r["tukey"].items()})
        if claim != "fig2d":
            lg = {k: np.log10(x[x > 0]) for k, x in g.items()}
            rl = sq.oneway(lg)
            print(f"  variant log10 (zeros left out: {sum(int((x == 0).sum()) for x in g.values())}): F = {rl['F']:.4f}, p = {rl['p']:.4f}")
print("\nPaper: upper F(2,31) = 12.07, p = 0.000133; left F = 0.02705, p = 0.973; front F = 1.326, p = 0.280")

**Result: fig2d-acute-upper-zone reproduced.** Leaving out the ninth acute-OA squid of the file
(59.6% upper, 1.3% left, 6.7% front) gives the paper's three sums of squares (25397.6, 42.9,
2436.7), and with it means of 30.0, 86.0 and 14.7%, F(2,31) = 12.07, p = 0.00013; Tukey separates
acute-OA from control and chronic-OA, not control from chronic-OA. With all 9 acute-OA squid the
acute-OA mean is 83.1% and the pattern is the same. The squid is chosen by matching the paper's
sums of squares, which fits to the target; two things support the choice. It matches all three
sums of squares at once, and in the workspace's figure tables its value is the one acute-OA zone
value with no distance beside it. If the rows of the zone table (F2D) and the distance table (F2C)
are the same squid in the same order, which is assumed, as no identifier links them, it is the
squid the distance and velocity files lack.

**Result: fig2e-left-right-unchanged reproduced.** Means 55.8, 54.1 and 57.1%; F(2,31) = 0.027,
p = 0.97 (the paper's values; with all 9, F = 0.27, p = 0.77).

**Result: fig2f-front-back-unchanged reproduced.** Means 58.3, 80.0 and 66.1%; F(2,31) = 1.326, p =
0.28 (with all 9, F = 0.52, p = 0.60).

## fig2g-attack-willingness

> Acidified conditions reduced the proportion of squid that attacked prey (logistic regression, pH
> p = 0.0422) with no effect of exposure duration (p = 0.2461); acute-OA and chronic-OA squid were
> 64.7% and 42.3% less likely to attack than controls.

Logistic regression (binomial GLM, logit) of attack on pH and exposure time, readings (a) to (c) of
the criterion. Reading (c) uses the measured tank pH of Supplementary Table 5 (8.20, 7.81, 7.80),
paper values used as covariate codes. The per-squid file holds 11 chronic-OA squid where the paper
and the workspace's Figure 2G table (`F2G`) have 22 (next cell). `F2G` gives the paper's counts, and
with two terms and three groups the model is saturated: it depends on the data only through the
counts of attackers and squid per group. The main test therefore fits per-squid outcomes rebuilt from
the `F2G` counts, which is exact for this model. The per-squid file as deposited is a variant.

In [ ]:
pc = csvs["Squid_Prey_Capture.csv"]
pc = pc.rename(columns={pc.columns[0]: "attack", pc.columns[2]: "hunt_s", pc.columns[3]: "attempts"})
counts = pc.groupby("Group").attack.agg(["sum", "count"]).reindex(sq.GROUPS)
print("Squid_Prey_Capture.csv:")
print(counts.rename(columns={"sum": "attacked", "count": "squid"}).to_string())
f2g = fig_tables["F2G"]
print("\nTable F2G of the R workspace (Figure 2G):")
print(f2g.to_string(index=False))
n_att = {g: int(round(m * n / 100)) for g, m, n in zip(f2g.group, f2g.Mean, f2g.N)}
n_tot = dict(zip(f2g.group, f2g.N.astype(int)))
print("Attackers implied by F2G (Mean% x N):", {g: f"{n_att[g]} of {n_tot[g]}" for g in sq.GROUPS})

In [ ]:
codings = {
    "(a) nominal pH, days in condition 90/7/90": ({"Control": 8.2, "Acute-OA": 7.8, "Chronic-OA": 7.8}, {"Control": 90, "Acute-OA": 7, "Chronic-OA": 90}),
    "(b) nominal pH, days of acidification 0/7/90": ({"Control": 8.2, "Acute-OA": 7.8, "Chronic-OA": 7.8}, {"Control": 0, "Acute-OA": 7, "Chronic-OA": 90}),
    "(c) tank pH 8.20/7.81/7.80, days 90/7/90": ({"Control": 8.20, "Acute-OA": 7.81, "Chronic-OA": 7.80}, {"Control": 90, "Acute-OA": 7, "Chronic-OA": 90}),
    "(c) tank pH 8.20/7.81/7.80, days 0/7/90": ({"Control": 8.20, "Acute-OA": 7.81, "Chronic-OA": 7.80}, {"Control": 0, "Acute-OA": 7, "Chronic-OA": 90}),
}

def fit_codings(data, title):
    print(f"=== {title}")
    for label, (ph, days) in codings.items():
        d = data.assign(pH=data.Group.map(ph), time=data.Group.map(days))
        fit = smf.glm("attack ~ pH + time", data=d, family=sm.families.Binomial()).fit()
        print(f"\n{label}")
        print(fit.summary2().tables[1][["Coef.", "Std.Err.", "z", "P>|z|"]].round(6).to_string())
        print(f"null deviance {fit.null_deviance:.2f} on {int(fit.df_model + fit.df_resid)} df, residual {fit.deviance:.2f} on {int(fit.df_resid)} df, AIC {fit.aic:.2f}, ROC area {sq.auc(d.attack.to_numpy(), fit.fittedvalues.to_numpy()):.4f}")

# Main: per-squid outcomes rebuilt from the F2G counts (exact for this saturated model).
rec = pd.DataFrame([{"Group": g, "attack": int(i < n_att[g])} for g in sq.GROUPS for i in range(n_tot[g])])
print("Reduction against control:", {g: round(100 * (1 - (n_att[g] / n_tot[g]) / (n_att["Control"] / n_tot["Control"])), 1) for g in sq.GROUPS[1:]})
fit_codings(rec, "counts of table F2G (" + ", ".join(f"{n_att[g]} of {n_tot[g]}" for g in sq.GROUPS) + ")")
print("\nPaper (Supplementary Table 4): pH 2.734246 (SE 1.345676, p = 0.0422), time 0.008351 (SE 0.00762, p = 0.2461); residual deviance 127.58; ROC area 0.6992")

In [ ]:
# Variant: the per-squid file as deposited (11 chronic-OA squid).
prop = counts["sum"] / counts["count"]
print("Reduction against control:", {g: round(100 * (1 - prop[g] / prop["Control"]), 1) for g in sq.GROUPS[1:]})
fit_codings(pc, "Squid_Prey_Capture.csv as deposited")

**Result: reproduced.** On the counts of the workspace's Figure 2G table, 29 of 46, 8 of 36 and 8
of 22 (the paper's), reading (a) gives the paper's model: pH 2.734 (p = 0.042), time 0.00835 (p =
0.246), residual deviance 127.58, ROC area 0.699, intercept -22.64; the reductions are 64.8% and
42.3%. Readings (b) and (c) give the same deviances with other coefficients (printed above). The per-squid file holds only 11 chronic-OA squid, 8 of
them attackers; on it the chronic-OA reduction is -15.4%, and under reading (a) the pH term is not
significant (coefficient -1.12, p = 0.55) while the time term is (p = 0.0045). The result rests on
the counts of the workspace's table; the per-squid file lacks 11 chronic-OA non-attackers (finding).

## fig2h-hunt-duration

> Acute-OA squid took about 2.5 times longer to capture prey than controls (262.2 against 106.7 s,
> p = 0.03); chronic exposure did not affect hunt duration.

One-way ANOVA on log10 durations and Tukey's test (Supplementary Table 4), squid that attacked.

In [ ]:
hunt = sq.by_group(pc[pc.attack == 1], "hunt_s")
print(sq.describe(hunt).round(1).to_string())
r = sq.oneway({g: np.log10(x) for g, x in hunt.items()})
print(f"\nANOVA on log10: F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.4f}; SS {r['SS_treat']:.3f} and {r['SS_resid']:.3f}; eta^2 = {r['eta2']:.3f}")
print("Paper: F(2,42) = 3.372, p = 0.0439; SS 2.374 and 14.79")
print("Tukey adjusted p (log10):", {k: round(x, 4) for k, x in r["tukey"].items()})
rn = sq.oneway({g: np.log(x) for g, x in hunt.items()})
print(f"Natural log: SS {rn['SS_treat']:.3f} and {rn['SS_resid']:.3f}, F = {rn['F']:.3f}")
raw = sq.oneway(hunt)
print(f"Variant, untransformed: F = {raw['F']:.3f}, p = {raw['p']:.4f}; Tukey", {k: round(x, 4) for k, x in raw["tukey"].items()})
print(f"Fold change acute/control: {hunt['Acute-OA'].mean() / hunt['Control'].mean():.2f}")

**Result: reproduced.** Means 106.7 s (control) and 262.2 s (acute-OA), 2.46-fold; the ANOVA on log10
durations gives F(2,42) = 3.372, p = 0.0439 with the paper's sums of squares; Tukey separates
acute-OA from control (p = 0.035) and chronic-OA (109.5 s) from neither (p = 0.72 and 0.32).

## fig2i-strike-attempts

> Most squid struck prey successfully within 1 to 2 attempts, strike attempts did not differ
> between groups, and every squid that attempted to hunt captured its prey.

In [ ]:
att = pc[pc.attack == 1]
frac = (att.attempts <= 2).groupby(att.Group).mean().reindex(sq.GROUPS)
print(f"Attacking squid: {len(att)}; with 1 or 2 attempts: {(att.attempts <= 2).sum()} ({100 * (att.attempts <= 2).mean():.1f}%)")
print("Per group:", frac.round(3).to_dict())
print("Attempts per group:", {g: np.bincount(att.loc[att.Group == g, "attempts"].astype(int)).tolist() for g in sq.GROUPS})
print("Squid with attempts but no capture (attack 0 with attempts recorded):", int(pc[(pc.attack == 0) & pc.attempts.notna()].shape[0]),
      "; captures with no duration:", int(att.hunt_s.isna().sum()))
kw = stats.kruskal(*[att.loc[att.Group == g, "attempts"] for g in sq.GROUPS])
print(f"Kruskal-Wallis H = {kw.statistic:.3f}, p = {kw.pvalue:.4f} (paper: 2.568, p = 0.2769)")

**Result: reproduced.** 38 of 45 attacking squid (84.4%) needed 1 or 2 attempts (control 89.7%,
acute-OA and chronic-OA 75%); Kruskal-Wallis H =
2.568, p = 0.277, the paper's values. The file codes squid as captured (1) or not (0), and every
squid with recorded attempts has a capture time; it has no record of squid that struck and missed
for good, so "every squid that attempted captured" holds as far as the file can show it.

## sfig3-prey-velocity-unchanged

> The velocity of the prey shrimp was not affected by CO2 treatment, time (10 and 20 min) or their
> interaction.

Only table `S3` of the R workspace holds these values. Its CO2 10 to 20 min column is a copy of the
control 10 to 20 min column (next cell), so the CO2 shrimp at 20 min are not in the deposit. The
criterion's two-way ANOVA (pH by time; Methods, Statistics) needs them: its time and interaction
terms cannot be tested, and a two-way ANOVA on the table as deposited takes the copy as data and
builds in "no effect" at 10 to 20 min. What the deposit allows is tested: the pH effect at 0 to 10
min (ordinary one-way ANOVA, with Welch's t and Mann-Whitney as variants) and, within the control
shrimp, the time effect (0 to 10 against 10 to 20 min; unpaired, as no shrimp identifier links
the columns). The two-way ANOVA on the table as deposited is printed last, only for reference.

In [ ]:
s3 = fig_tables["S3"]
print(s3.round(4).to_string())
print("\nNon-missing per column:", s3.notna().sum().to_dict())
a, b = s3["pH 8.1 10-20 Min"].dropna().to_numpy(), s3["pH 7.8 10-20 Min"].dropna().to_numpy()
print("pH 8.1 10-20 Min and pH 7.8 10-20 Min identical:", len(a) == len(b) and bool(np.array_equal(a, b)))

In [ ]:
x0, x1 = s3["pH 8.1 0-10 Min"].dropna(), s3["pH 7.8 0-10 Min"].dropna()
c2 = s3["pH 8.1 10-20 Min"].dropna()
print(f"0-10 min: control {x0.mean():.3f} cm/s (n {len(x0)}), CO2 {x1.mean():.3f} (n {len(x1)})")
f = stats.f_oneway(x0, x1)
print(f"  pH at 0-10 min, one-way ANOVA F(1,{len(x0) + len(x1) - 2}) = {f.statistic:.3f}, p = {f.pvalue:.3f}; "
      f"Welch t p = {stats.ttest_ind(x0, x1, equal_var=False).pvalue:.3f}; Mann-Whitney p = {stats.mannwhitneyu(x0, x1).pvalue:.3f}")
f = stats.f_oneway(x0, c2)
print(f"Control shrimp, 0-10 against 10-20 min: {x0.mean():.3f} against {c2.mean():.3f} cm/s; one-way ANOVA F(1,{len(x0) + len(c2) - 2}) = {f.statistic:.3f}, p = {f.pvalue:.3f}; "
      f"Mann-Whitney p = {stats.mannwhitneyu(x0, c2).pvalue:.3f}")
print("CO2 shrimp at 10-20 min, and so the interaction: not in the deposit (the column is a copy of control).")

# For reference only: the two-way ANOVA on the table as deposited, which takes the copied column as data.
long = s3.melt(var_name="cond", value_name="velocity").dropna()
long["pH"] = long.cond.str.slice(3, 6)
long["time"] = long.cond.str.slice(7)
print("\nFor reference, the table as deposited:")
print(long.groupby(["pH", "time"]).velocity.agg(["count", "mean", "sem"]).round(4).to_string())
fit = smf.ols("velocity ~ C(pH, Sum) * C(time, Sum)", data=long).fit()
print("two-way ANOVA as deposited, type III:")
print(sm.stats.anova_lm(fit, typ=3).round(4).to_string())

**Result: partially reproduced.** The part the deposit allows holds: at 0 to 10 min control (0.224
cm/s, 8 shrimp) and CO2 (0.170 cm/s, 11 shrimp) shrimp do not differ (one-way ANOVA F(1,17) = 0.23, p = 0.64; Welch t p = 0.65), and the control shrimp
do not differ between 0 to 10 and 10 to 20 min (0.224 against 0.422
cm/s, F(1,14) = 0.88, p = 0.36). The CO2 shrimp at 10 to 20 min are not in
the deposit (their column is a copy of the control column), so the time effect in the CO2 group and
the interaction cannot be tested. The two-way ANOVA on the table as deposited (no term below p =
0.05) takes the copy as data and is not evidence for the claim.

## fig3c-ratio-declines

> As stimulus frequency increased, the ratio of maximum response frequency to stimulus frequency
> decreased in all three groups, markedly above 40 Hz.

The deposit holds the group mean (and SD) of the ratio per stimulus frequency, not the per-squid
spectra, so the criterion is applied to these deposited means. Frequencies with no value in a group
are left out of that group.

In [ ]:
erg = csvs["Squid_ERG.csv"]
wide = erg.pivot(index="Frequency", columns="Group", values="mean_value")[sq.GROUPS]
print(wide.round(3).to_string())
print("\nRows with an SD (more than one squid):", erg.sd_value.notna().sum(), "of", len(erg), "; the file does not say why the others have none")
for g in sq.GROUPS:
    s = wide[g].dropna()
    ref = s.loc[20]
    above = s[s.index > 40]
    not_below = above[above >= ref]
    hi = s.index.max()
    print(f"\n{g}: ratio at 20 Hz {ref:.4f}; frequencies above 40 Hz with a value {list(above.index)}")
    print(f"  not below the 20 Hz value: {not_below.round(4).to_dict()}")
    print(f"  at the highest frequency ({hi} Hz) {s.loc[hi]:.4f}; lowest {s.min():.4f} at {s.idxmin()} Hz; difference {s.loc[hi] - s.min():.4f}")
    print(f"  mean ratio up to 50 Hz {s[s.index <= 50].mean():.3f}, above 50 Hz {s[s.index > 50].mean():.3f}")
# How many squid lie behind each mean? The file gives no n. An SD of 0 marks squid that all gave the same
# ratio; a missing SD is not explained.
print("\nSD per group and frequency (NaN: no SD given):")
print(erg.pivot(index="Frequency", columns="Group", values="sd_value")[sq.GROUPS].round(3).to_string())
# Peak frequency implied by each mean ratio, in bins of an FFT at the paper's 4 kHz sampling (Methods, ERG
# recording) with 2048 points (1.953 Hz), the bin width these values fit; the paper gives no FFT length.
bin_hz = 4000 / 2048
print(f"\nImplied peak frequency (ratio x stimulus frequency) in bins of {bin_hz:.4f} Hz:")
print(wide.mul(wide.index.to_series(), axis=0).div(bin_hz).round(3).to_string())

**Result: not reproduced.** In no group is the mean ratio below its 20 Hz value at every frequency
above 40 Hz: control is at 0.998 (45 Hz) and 0.990 (50 Hz) against 0.977 at 20 Hz, acute-OA at 0.998
(45 Hz) and 1.062 (57.5 Hz), chronic-OA at 1.039 (47 Hz). In control the ratio stays near 1 up to
50 Hz and lies between 0.28 and 0.51 above it. In acute-OA it stays near 1 up to 45 Hz, is 0.79 at
50 Hz and lies between 0.30 and 0.47 above 50 Hz except at 57.5 Hz (1.062). Chronic-OA falls at 40
and 42 Hz (0.39, 0.37) and returns near 1 at 47 Hz. The ratio at the highest frequency (65 Hz) is the lowest
in acute-OA (0.300) but not within 0.1 of the lowest in control (0.391 against 0.284 at 62.5 Hz)
or chronic-OA (0.361 against 0.252). The decline the statement describes
is present above 50 Hz in the deposited means; the criterion's "every frequency above 40 Hz" is not met.

The ratios up to 45 Hz in control and acute-OA are whole numbers of FFT bins of 1.953 Hz (4 kHz
sampling, 2048 points): 0.977 at 10 and 20 Hz is a peak at 5 and 10 bins (9.77 and 19.53 Hz), 1.042
at 30 Hz is 16 bins, 1.025 at 40 Hz 21 bins and 0.998 at 45 Hz 23 bins. The two groups have the
same mean at each of these frequencies. Where an SD is given there it is 0 (acute-OA at 10, 40 and
45 Hz, control at 45 Hz), so every squid behind that mean peaked in the same bin. At the other
frequencies up to 40 Hz the SD is missing, and the file does not show whether such a mean rests on
one squid or on several. The 20 Hz reference is 0.023 below 1 by this rounding alone, so a frequency
whose nearest bin lies just above it (45 Hz, 0.998) is not below it although the response follows
the stimulus. Part of the failure is this quantization, which the criterion does not allow for; a
question has been filed (`questions/allen-2026.yaml`). The other failures are not rounding to the
nearest bin: chronic-OA at 47 Hz is a peak at 25 bins, one above the bin nearest the stimulus, and
acute-OA at 57.5 Hz is not a whole number of bins. The file gives no n per mean, so the paper's n =
8, 8 and 9 cannot be checked.

## sfig4a-amplitude-attenuation

Blocked, data missing: the ERG traces and spectra are not deposited; `Squid_ERG.csv` holds only the
ratio of maximum response frequency to stimulus frequency.

## fig3d-cff-unchanged

> Critical flicker fusion frequency did not differ between treatments (control 50.8, acute-OA 50.8,
> chronic-OA 47.7 Hz).

Reading (b) of the criterion: the CFF the deposit gives per squid (3, 3 and 6 squid). Reading (a),
CFF for every squid from its ratios, cannot be run, as the per-squid ratios are not deposited. As a
reference, the highest frequency at which the deposited group mean ratio is at least 0.9 is printed.
One-way ANOVA at p < 0.05, with Tukey's test.

In [ ]:
cff_df = csvs["Squid_CFF.csv"]
cff = sq.by_group(cff_df, cff_df.columns[0])
d = sq.describe(cff)
print(d.round(3).to_string())
r = sq.oneway(cff)
print(f"\nANOVA F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.4f}; eta^2 = {r['eta2']:.3f}")
print("Tukey adjusted p:", {k: round(x, 4) for k, x in r["tukey"].items()})
kw = stats.kruskal(*cff.values())
print(f"Variant, Kruskal-Wallis H = {kw.statistic:.3f}, p = {kw.pvalue:.4f}")
for g in sq.GROUPS:
    s = wide[g].dropna()
    print(f"{g}: highest frequency with group mean ratio >= 0.9: {s[s >= 0.9].index.max()} Hz")

**Result: reproduced.** Means 50.7, 50.7 and 47.7 Hz (n = 3, 3, 6); one-way ANOVA F(2,9) = 0.541, p
= 0.60 (η² = 0.11), so no difference is found; Tukey p ≥ 0.68 for every pair.

## fig3d-cff-variance

> The standard deviation of CFF was 4 to 5 times higher in the acidified groups than in controls.

Sample SDs (n - 1) of the deposited CFF per squid.

In [ ]:
sd = d["sd"]
print("SD:", sd.round(3).to_dict())
print(f"SD ratio acute-OA/control {sd['Acute-OA'] / sd['Control']:.2f}, chronic-OA/control {sd['Chronic-OA'] / sd['Control']:.2f} (criterion: 3.6 to 5.5)")
lev = stats.levene(*cff.values(), center="median")
print(f"Not judged: Brown-Forsythe test of equal variances, p = {lev.pvalue:.3f}")

**Result: reproduced.** SDs 1.15, 6.03 and 5.47 Hz; ratios 5.22 (acute-OA) and 4.73 (chronic-OA),
both in 3.6 to 5.5. With 3 control squid the control SD rests on three values; a Brown-Forsythe test
does not find the variances to differ (p = 0.61), which is not judged.

## rnaseq-assembly

> The de novo transcriptome assembly (...) had 902,413 transcripts, an N50 of 476 bp overall and
> 2038 bp for the top 78% of expressed transcripts, read mapping rates above 97% (mean 99.51%), and
> 69,368 transcripts after expression filtering.

The transcript count is that of the deposited RSEM matrix; E78N50 and N50 are read from the
deposited `ExN50.stats` (Trinity's contig_ExN50_statistic.pl output, `Bash_scripts.sh`), as the
assembly FASTA is not in the dataset. The count file's header names 16 samples while its rows hold
15 values; the cell matches its columns to the workspace's `rawMatrix`, the 15 optic lobe samples
the script used. The filter is applied to `rawMatrix` as the deposited `R_scritps.Rmd` does it
(reading (a): rows with a non-zero mean, rounded, then at least 10 counts in at least 3 samples);
reading (b), the 12 samples of the three groups alone, and the file's own 14 optic lobe columns
are variants. Mapping rates are not deposited. The number of sequences of the public assembly, NCBI
TSA GLBY00000000.1 (the claims file's `where`), is read from its master record as a variant of the
transcript count.

In [ ]:
rmd = sq.zip_text("R_scritps.Rmd").splitlines()
print("How the authors' script (squid.zip, R_scritps.Rmd) reads the matrix:")
for n in (57, 59):
    print(f"  line {n}: {rmd[n - 1].strip()}")
header, n_rows = sq.count_matrix_rows()
print("RSEM.isoform.counts.matrix columns:", header)
print("transcripts (rows):", n_rows)
exn50 = pd.read_csv(pd.io.common.StringIO(sq.zip_text("ExN50.stats")), sep="\t")
print(exn50.columns.tolist(), len(exn50), "rows")
print("E78N50:", exn50.loc[exn50.Ex == 78, "ExN50"].item(), " maximum ExN50 from Ex 25 on:", exn50.loc[exn50.Ex >= 25, "ExN50"].max(),
      " N50 at Ex 100:", round(exn50.loc[exn50.Ex == 100, "ExN50"].item(), 1), " num_genes at Ex 100:", exn50.loc[exn50.Ex == 100, "num_genes"].item())
with __import__("zipfile").ZipFile(sq.STAGED / "squid.zip") as z, z.open(sq.ZIP_DIR + "RSEM.isoform.counts.matrix") as f:
    f.readline()
    first = f.readline().decode().rstrip("\n").split("\t")
print(f"header names {len(header)} samples; a data row holds an id and {len(first) - 1} values")
ws_raw = sq.matrix(sq.read_rdata({"rawMatrix"})["rawMatrix"])
ws_raw.index = ["TRINITY_" + i for i in ws_raw.index]
print("Workspace rawMatrix (the script's 15 optic lobe samples):", ws_raw.shape, list(ws_raw.columns))
for how in ("r", "first"):
    cm = sq.read_count_matrix(how).reindex(ws_raw.index)
    match = {wc: next((fc for fc in cm.columns if np.array_equal(cm[fc].to_numpy(), ws_raw[wc].to_numpy())), None) for wc in ws_raw.columns}
    print(f"\nFile columns named as {'R read.csv(row.names = 1) does' if how == 'r' else 'the first 15 header names'}; workspace column -> identical file column:")
    print(match)
    del cm

In [ ]:
raw = ws_raw
kept_mean = raw[raw.mean(axis=1) != 0]
rounded = np.round(kept_mean)
filt_a = int(((rounded >= 10).sum(axis=1) >= 3).sum())
filt_b = int(((kept_mean.sum(axis=1) >= 10) & ((kept_mean > 0).sum(axis=1) >= 3)).sum())
grp12 = [c for c in raw.columns if not c.startswith("Re_")]
r12 = np.round(raw[grp12][raw[grp12].mean(axis=1) != 0])
filt_12 = int(((r12 >= 10).sum(axis=1) >= 3).sum())
counts = sq.read_count_matrix("first")
fr = counts[[c for c in counts.columns if "OL" in c]]
fr = np.round(fr[fr.mean(axis=1) != 0])
print(f"\nOn the workspace's 15 optic lobe samples; rows with non-zero mean {len(kept_mean)}")
print(f"(a) at least 10 counts in at least 3 of 15 samples: {filt_a}")
print(f"(b) total >= 10 and detected in >= 3 samples: {filt_b}")
print(f"variant, (a) on the 12 samples of the three groups: {filt_12}")
print(f"variant, (a) on the file's 14 optic lobe columns (Re_8.1_OL-3 missing): {int(((fr >= 10).sum(axis=1) >= 3).sum())}")
fa = np.round(counts[counts.mean(axis=1) != 0])
print(f"variant, (a) on all 15 value columns, which read.csv(row.names = 1) names as the 15 optic lobe samples: {int(((fa >= 10).sum(axis=1) >= 3).sum())}")
tsa = sq.tsa_summary()
print(f"NCBI TSA {tsa['accessionversion']} ({tsa['title']}): {tsa['slen']:,} sequences, against {n_rows:,} transcripts in the count matrix "
      f"({100 * (tsa['slen'] / n_rows - 1):.1f}%)")
res_n = {k: len(sq.s4_dataframe(v)) for k, v in zip(ws["resNormal"].names(), ws["resNormal"].value)}
print("Rows of the deposited DESeq2 results (resNormal):", res_n)
del counts, fr, fa, raw, kept_mean, rounded, r12, ws_raw

**Result: partially reproduced; the mapping rate is not in the dataset.** Four of the five values are
within tolerance: 902,413 transcripts, E78N50 2038 bp and N50 475.5 bp (476) in the deposited
statistics, and 69,368 transcripts after the filter of the deposited script on the workspace's
`rawMatrix` (reading (a)), the row count of the authors' DESeq2 results. The mean mapping rate cannot
be tested. The N50 of 475.5 bp is the ExN50 at Ex 100, which Trinity computes over the 610,324
entries with expression (the file's `num_genes`), not over all 902,413 transcripts; it is probably
how the paper got 476, but the N50 of the whole assembly needs the assembly FASTA, which is not
deposited. The public TSA assembly holds 870,834 sequences, 3.5% fewer than the count matrix and
within the 10% tolerance. The deposited count file cannot give the filtered count by itself: its 15
value columns are the workspace's gill sample and the first 14 optic lobe samples, and Re_8.1_OL-3
is missing; read with the script's `read.csv(row.names = 1)`, every name is shifted by one sample.

## fig4a-pca

> In a PCA of optic lobe transcriptomes, acute-OA samples lay near controls along PC1 (35% of
> variance), while chronic-OA samples diverged along PC1 and PC2 (15%).

The authors' objects: `vstsva.adj.limma` is the VST of the DESeq2 fit with design ~SV1 + SV2 +
condition, with the two surrogate variables of `svseq` removed by limma's removeBatchEffect
(reading (a); `R_scritps.Rmd`, "Limma adjust for the visualization"). The cell first checks that
the deposited matrix equals that removal applied to `vstsva.adj`. Reading (b), the known batches
removed the same way, is applied to `vst` (the VST of the fit with design ~condition). PCA as
DESeq2's plotPCA (500 most variable rows, centred, unscaled). The script runs plotPCA on all 15
samples and keeps the 12 of the three groups (`x[1:12,]`), and also calls it on the 12 alone; both
are shown. The criterion does not define the control "spread"; it is taken as the range of the
control samples on each PC (an assumption, see the result), and the SD is shown as the other reading.
Reading (b) is also applied to `vstsva.adj`, the VST of reading (a), to check whether its result
comes from the `vst` object.

In [ ]:
coldata = sq.df_from(ws["df"])
limma_mat = sq.s4_assay(ws["vstsva.adj.limma"])
adj_mat = sq.s4_assay(ws["vstsva.adj"])
vst_mat = sq.s4_assay(ws["vst"])
print("VST matrices (rows x samples):", limma_mat.shape, adj_mat.shape, vst_mat.shape)
print("samples:", list(limma_mat.columns))
print(coldata.to_string())
sv = sq.matrix(ws["svseq"]["sv"]).to_numpy()
print("surrogate variables:", sv.shape[1], "(n.sv =", int(ws["svseq"]["n.sv"].value[0]), ")")
conds = pd.Categorical(coldata["condition"], categories=["Ct_8.1", "pH_7.8", "pH_7.8C", "Re_8.1"])
design = pd.get_dummies(conds, drop_first=True).astype(float).to_numpy()
design = np.column_stack([np.ones(len(conds)), design])
check = sq.remove_batch_effect(adj_mat, design, sv)
print("max |deposited limma matrix - removeBatchEffect(vstsva.adj, ~condition, SVs)|:", float(np.abs(check.to_numpy() - limma_mat.to_numpy()).max()))
batch = np.where(coldata["batch"].astype(str).to_numpy() == "1", 1.0, -1.0)[:, None]   # a factor ("1", "2"); contr.sum coding, as removeBatchEffect(batch=)
mat_b = sq.remove_batch_effect(vst_mat, design, batch)
label = {"Ct_8.1": "Control", "pH_7.8": "Acute-OA", "pH_7.8C": "Chronic-OA", "Re_8.1": "Re_8.1"}
groups15 = coldata["condition"].map(label).to_numpy()

In [ ]:
def pca_summary(mat, cols, name):
    scores, frac = sq.plotpca(mat[cols])
    g = pd.Series(groups15, index=mat.columns)[cols]
    sc = scores.loc[cols, ["PC1", "PC2"]]
    cen = sc.groupby(g.to_numpy()).mean()
    ctrl = sc[g.to_numpy() == "Control"]
    spread = ctrl.max() - ctrl.min()
    keep = g.to_numpy() != "Re_8.1"
    out = {"reading": name, "PC1 %": round(100 * frac[0], 1), "PC2 %": round(100 * frac[1], 1),
           "|acute-control| PC1": abs(cen.loc["Acute-OA", "PC1"] - cen.loc["Control", "PC1"]),
           "|chronic-control| PC1": abs(cen.loc["Chronic-OA", "PC1"] - cen.loc["Control", "PC1"]),
           "|chronic-control| PC2": abs(cen.loc["Chronic-OA", "PC2"] - cen.loc["Control", "PC2"]),
           "control range PC1": spread["PC1"], "control range PC2": spread["PC2"]}
    return out, sc[keep].assign(group=g.to_numpy()[keep])

all15 = list(limma_mat.columns)
first12 = all15[:12]
rows, plots = [], {}
for name, mat, cols in [("(a) SVs removed, PCA on 15 (script)", limma_mat, all15), ("(a) SVs removed, PCA on 12", limma_mat, first12),
                        ("(b) batch removed, PCA on 15", mat_b, all15), ("(b) batch removed, PCA on 12", mat_b, first12),
                        ("(b) batch removed from vstsva.adj, PCA on 12", sq.remove_batch_effect(adj_mat, design, batch), first12)]:
    o, sc = pca_summary(mat, cols, name)
    rows.append(o)
    plots[name] = sc
pca_tab = pd.DataFrame(rows).set_index("reading")
pca_tab["acute nearer control on PC1"] = pca_tab["|acute-control| PC1"] < pca_tab["|chronic-control| PC1"]
pca_tab["chronic separated PC1"] = pca_tab["|chronic-control| PC1"] > pca_tab["control range PC1"]
pca_tab["chronic separated PC2"] = pca_tab["|chronic-control| PC2"] > pca_tab["control range PC2"]
for name, sc in plots.items():
    ctrl = sc[sc.group == "Control"]
    pca_tab.loc[name, "control SD PC1"] = ctrl.PC1.std()
    pca_tab.loc[name, "control SD PC2"] = ctrl.PC2.std()
    chron = sc[sc.group == "Chronic-OA"]
    pca_tab.loc[name, "chronic outside control range PC2"] = int(((chron.PC2 > ctrl.PC2.max()) | (chron.PC2 < ctrl.PC2.min())).sum())
pca_tab["chronic separated PC2 (SD reading)"] = pca_tab["|chronic-control| PC2"] > pca_tab["control SD PC2"]
print(pca_tab.round(2).T.to_string())
print("\nPaper: PC1 35%, PC2 15%")
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, name in zip(axes, ["(a) SVs removed, PCA on 15 (script)", "(a) SVs removed, PCA on 12"]):
    sc = plots[name]
    for g, c in zip(sq.GROUPS, ["tab:green", "tab:orange", "tab:purple"]):
        ax.scatter(sc.loc[sc.group == g, "PC1"], sc.loc[sc.group == g, "PC2"], label=g, color=c)
    ax.set_title(name, fontsize=9); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("All points are within the axes.")
print(plots["(a) SVs removed, PCA on 12"].round(2).to_string())
print("\nReading (b), PCA on 12, scores:")
print(plots["(b) batch removed, PCA on 12"].round(1).to_string())
print("Lowest value of each VST matrix: vst", round(float(vst_mat.to_numpy().min()), 4), "; vstsva.adj", round(float(adj_mat.to_numpy().min()), 4))

**Result: partially reproduced.** On the authors' batch-corrected matrix (reading (a)), the PCA of
the 12 samples of the three groups gives PC1 35.0% and PC2 15.0%, the paper's values (the script's
PCA of all 15 samples gives 29.0% and 13.5%). Acute-OA lies near the controls on PC1 (centroid
distance 3.5 against 62.9 for chronic-OA), and chronic-OA is separated from the controls on PC1
(62.9 against a control range of 13.4). On PC2 the chronic-OA centroid is 18.0 from the control
centroid, less than the control range (29.6) and more than the control SD (12.8); 1 of the 3
chronic-OA samples lies outside the control range. The outcome rests on the reading of "spread",
which the criterion does not define: under the range, taken here, PC2 separation fails and the
claim is partially reproduced; under the SD it would be reproduced. The range is taken because the
paper speaks of "significant divergence" along PC2, and two of the three chronic-OA samples lie
among the controls on it. A question on this has been filed (`questions/allen-2026.yaml`).
Reading (b), the known batches removed: on the PCA of the 12 samples PC1 is 31.7% and PC2 16.6%,
just outside the PC2 tolerance (13.5 to 16.5%), and chronic-OA is 15.7 from the controls on PC2
against a control range of 63.3 and SD of 27.2, so it is not separated on PC2 under either reading
of spread; on all 15 samples PC1 is 25.7%. Reading (b) does not meet the criterion. (In the first
version of this notebook the batch factor, stored as text, was compared with a number, so no batch
was removed and PC1 came out above 90%; this is corrected.)

## sfig5b-acute-clusters-with-control

> After batch-effect removal, acute-OA samples clustered closer to controls than chronic-OA samples
> did.

Euclidean distances between the 12 samples on the batch-corrected matrix, all rows, as the
script's "Supplementary Figure S5A" chunk computes them (`dist(t(assay(vstsva.adj.limma)[,1:12]))`);
reading (b) on the batch-removed matrix as a variant.

In [ ]:
from scipy.spatial.distance import pdist, squareform
for name, mat in [("(a) SVs removed", limma_mat), ("(b) batch removed", mat_b)]:
    x = mat[first12]
    dm = pd.DataFrame(squareform(pdist(x.T.to_numpy())), index=first12, columns=first12)
    g = pd.Series(groups15[:12], index=first12)
    ac = dm.loc[g == "Acute-OA", g == "Control"].to_numpy().mean()
    cc = dm.loc[g == "Chronic-OA", g == "Control"].to_numpy().mean()
    within = dm.loc[g == "Control", g == "Control"].to_numpy()[np.triu_indices(4, 1)].mean()
    print(f"{name}: mean distance acute-OA to control {ac:.1f}, chronic-OA to control {cc:.1f}, control to control {within:.1f}; max {dm.to_numpy().max():.1f}")

**Result: reproduced.** On the batch-corrected matrix the mean distance from acute-OA to control
samples is 187.7, from chronic-OA to control 255.2 (control to control 197.7); reading (b) gives the
same order (477.1 against 532.8). Its distances are 2.1 to 2.5 times larger (2.54 for acute-OA to
control, 2.09 for chronic-OA to control, 2.31 for control to control). It is computed on `vst`, whose
lowest value is 0.005 against 3.65 for `vstsva.adj`; why the distances differ by this much was not
examined. It is a variant only. (The first version of this notebook removed no batch in reading (b),
see fig4a-pca.)

## GSEA results (fig4b to fig4fgh, chronic-nachr-subunits-down, optic-lobe-no-ion-regulation)

GSEA and DESeq2 are not re-run: clusterProfiler, the authors' AnnotationForge GO database and R
are not in this Python environment. The authors' `gse.Normal` objects are read: gseGO results for
acute-OA (AvsCT) and chronic-OA (CvsCT) against control, run with the parameters of the Methods
(checked below against the objects' stored parameters). The outcomes of these claims therefore show
that the paper's statements agree with the authors' saved results; they are not an independent
recomputation from the counts. What is checked: each object's ranked gene list is the shrunken log2
fold change of the deposited DESeq2 results `resNormal`, and the DESeq2 fits in the workspace rest
on the deposited counts (their counts are the rounded `rawMatrix` rows that pass the filter, and
DESeq2's median-of-ratios size factors computed here from them equal the stored ones; next cells).

In [ ]:
fits = sq.read_rdata({"dds", "ddssva", "rawMatrix"})
raw = sq.matrix(fits["rawMatrix"])
def formula(o):
    """An R formula (language object) as text."""
    if o.type == "symbol":
        return o.value
    f, *args = [x for _, x in o.value]
    return f" {f.value} ".join(formula(a) for a in args) if len(args) == 2 else f.value + formula(args[0])
for k in ("dds", "ddssva"):
    o = fits[k]
    cnt = sq.matrix(o["assays"]["data"]["listData"].value[0])
    stored = sq.s4_dataframe(o["colData"])["sizeFactor"].to_numpy()
    lg = np.log(cnt.to_numpy().astype(float))
    with np.errstate(invalid="ignore"):
        ok = np.isfinite(lg).all(axis=1)
    sf = np.exp(np.median(lg[ok] - lg[ok].mean(axis=1, keepdims=True), axis=0))   # DESeq2's median-of-ratios
    print(f"{k}: design {formula(o['design'])}; counts {cnt.shape}; equal to round(rawMatrix) on these rows: "
          f"{np.array_equal(cnt.to_numpy(), np.round(raw.loc[cnt.index].to_numpy()))}; "
          f"size factors recomputed against stored, max |difference| {np.abs(sf - stored).max():.1e}")
print("stored size factors:", dict(zip(cnt.columns, stored.round(4))))
del fits, raw, cnt, lg

In [ ]:
gse = {k: v for k, v in zip(ws["gse.Normal"].names(), ws["gse.Normal"].value)}
res = {k: sq.s4_dataframe(v) for k, v in zip(ws["resNormal"].names(), ws["resNormal"].value)}
gres = {}
for k, o in gse.items():
    p = o["params"]
    print(k, "params:", {n: (x.value[0] if len(x.value) else None) for n, x in zip(p.names(), p.value)},
          "| setType", o["setType"].value, "| gene sets in object", len(o["geneSets"].value))
    gl = o["geneList"]
    gl = pd.Series(gl.value, index=gl.attr["names"].value)
    r = res[k].dropna(subset=["padj"])["log2FoldChange"]
    common = gl.index.intersection(r.index)
    print(f"  ranked list {len(gl)} genes; resNormal with padj {len(r)}; shared {len(common)}; max |difference| {np.abs(gl[common] - r[common]).max():.2e}")
    t = sq.df_from(o["result"])
    gres[k] = t
    print(f"  significant sets {len(t)} (max p.adjust {t['p.adjust'].max():.4f}); NES < 0: {(t.NES < 0).sum()}, NES > 0: {(t.NES > 0).sum()}")
    print("  by ontology and sign:", t.groupby(["ONTOLOGY", np.where(t.NES > 0, "up", "down")]).size().to_dict())
A, C = gres["AvsCT"], gres["CvsCT"]

## fig4b-shared-down-sets

> Only 2 of 36 GO gene sets were down-regulated under both acute and chronic acidification,
> organelle inner membrane and mitochondrial inner membrane.

Readings of "36" from the criterion, as amended after this attempt's question
(`questions/allen-2026.yaml`): (a) the sets significant in both treatments, in either direction
(the Venn intersect; the deposited script's Figure 4C chunk plots `intersect(CvsCT ID, AvsCT ID)`
with `showCategory = 36`), (b) the sets down-regulated in either treatment, (c) all sets of the Venn
diagram (significant in either treatment).

In [ ]:
downA, downC = set(A.ID[A.NES < 0]), set(C.ID[C.NES < 0])
upA, upC = set(A.ID[A.NES > 0]), set(C.ID[C.NES > 0])
both_down = downA & downC
print("Down in both:", len(both_down), A.set_index("ID").loc[sorted(both_down), "Description"].to_dict())
readings_36 = {"(a) significant in both (Venn intersect)": len(set(A.ID) & set(C.ID)),
               "(b) down in either": len(downA | downC),
               "(c) all sets in the Venn diagram (significant in either)": len(set(A.ID) | set(C.ID))}
for k, n in readings_36.items():
    print(f"{k}: {n}; within 10% of 36: {abs(n - 36) <= 3.6}")
print("Venn regions: A- only", len(downA - set(C.ID)), "| C- only", len(downC - set(A.ID)), "| A+ ", len(upA), "| C+", len(upC),
      "| A- and C+", len(downA & upC), "| A+ and C-", len(upA & downC), "| A+ and C+", len(upA & upC))


**Result: reproduced.** Exactly two sets are down-regulated in both treatments, organelle inner
membrane and mitochondrial inner membrane. Under reading (a), the sets significant in both
treatments number 36, the paper's count; readings (b) and (c) give 122 and 646. The claims file
added reading (a) in answer to this attempt's question; under the earlier readings, (b) and (c)
here, the claim was partially reproduced.

In [ ]:
def find(table, pattern, sign):
    t = table[table.Description.str.contains(pattern, case=False, regex=True)]
    t = t[t.NES < 0] if sign < 0 else t[t.NES > 0]
    return t[["ID", "ONTOLOGY", "Description", "NES", "p.adjust"]]

def family_table(table, families, sign):
    rows = []
    for fam, pat in families.items():
        hit = find(table, pat, sign)
        rows.append({"family": fam, "pattern": pat, "terms": len(hit),
                     "examples": "; ".join(f"{d} ({n:.2f})" for d, n in zip(hit.Description[:3], hit.NES[:3]))})
    return pd.DataFrame(rows)

## fig4c-acute-ridge

> Acute acidification suppressed gene sets of lipid, carboxylic acid and organic acid metabolism,
> transferase and endopeptidase complexes and ribosomal subunits, and up-regulated cell-cell
> adhesion.

Each family is matched on the term names (case-insensitive) of the acute-OA results, all of which
have BH-adjusted p < 0.05.

In [ ]:
fam_c = {"cellular lipid metabolic process": r"cellular lipid metabolic process", "carboxylic acid metabolic process": r"carboxylic acid metabolic process",
         "organic acid metabolic process": r"organic acid metabolic process", "transferase complex": r"transferase complex",
         "endopeptidase complex": r"endopeptidase complex", "ribosomal subunit": r"ribosomal subunit"}
t_down = family_table(A, fam_c, -1)
t_up = family_table(A, {"cell-cell adhesion": r"cell-cell adhesion"}, +1)
print(pd.concat([t_down.assign(direction="down"), t_up.assign(direction="up")]).to_string(index=False))
print("\nAll acute-OA up-regulated sets:", A.loc[A.NES > 0, ["Description", "NES", "p.adjust"]].round(4).to_dict("records"))

**Result: reproduced.** In the acute-OA results each named family has at least one down-regulated
term (cellular lipid metabolic process NES -1.39, carboxylic and organic acid metabolic process
-1.47, transferase complex -1.62, endopeptidase complex -1.92, three ribosomal subunit terms -1.75
to -2.24), and cell-cell adhesion is up (NES 1.45).

## fig4d-acute-tree

> Under acute acidification down-regulated clusters included mitochondrial and energy metabolism
> (respirasome, electron transport chain complex IV, mitochondrial envelope), fatty acid
> biosynthesis and elongase activity, organelle and protein localization, and ncRNA
> ribonucleoprotein and 3'-end RNA processing.

Matched in the acute-OA down-regulated sets; the script's tree uses those not significant in
chronic-OA, shown as a variant.

In [ ]:
fam_d = {"respirasome": r"respirasome", "complex IV": r"cytochrome[- ]c oxidase|complex IV", "mitochondrial envelope": r"mitochondrial envelope",
         "fatty acid biosynthesis or elongase": r"fatty acid biosynthe|elongase", "organelle or protein localization": r"organelle localization|protein localization",
         "ncRNA ribonucleoprotein or 3'-end processing": r"ribonucleoprotein|3'-end"}
print(family_table(A, fam_d, -1).to_string(index=False))
a_only = A[~A.ID.isin(C.ID)]
print("\nVariant, acute-OA down and not significant in chronic-OA (script's Fig. 4D), sets:", int((a_only.NES < 0).sum()))
print(family_table(a_only, fam_d, -1)[["family", "terms"]].to_string(index=False))

**Result: reproduced.** Every named family has down-regulated acute-OA terms (respirasome, complex
IV, mitochondrial envelope, fatty acid biosynthesis and elongase activity, protein localization,
RNA 3'-end processing and ribonucleoprotein complex biogenesis), also within the 51 sets of the
script's tree.

## fig4e-chronic-synaptic-down

> Under chronic acidification down-regulated gene sets included postsynaptic membrane, synaptic
> membrane, transmitter-gated channel activity and neurotransmitter receptor activity, NAD(P)H
> dehydrogenase activity, respiratory electron transport, cell adhesion molecules and prefoldin
> complex.

In [ ]:
fam_e = {"postsynaptic membrane": r"^postsynaptic membrane$", "synaptic membrane": r"^synaptic membrane$",
         "transmitter-gated channel activity": r"transmitter-gated", "neurotransmitter receptor activity": r"neurotransmitter receptor activity",
         "NAD(P)H dehydrogenase": r"NAD\(P\)H dehydrogenase|NADH dehydrogenase", "respiratory electron transport chain": r"respiratory electron transport",
         "cell adhesion": r"cell adhesion", "prefoldin complex": r"prefoldin"}
print(family_table(C, fam_e, -1).to_string(index=False))
print("\nAll chronic-OA down-regulated sets:")
print(C.loc[C.NES < 0, ["ONTOLOGY", "Description", "NES", "p.adjust"]].sort_values("NES").round(4).to_string(index=False))

**Result: reproduced.** Among the 40 chronic-OA down-regulated sets are postsynaptic membrane (NES
-1.91), synaptic membrane (-1.92), transmitter-gated channel activity (-1.80), neurotransmitter
receptor activity (-1.83), NAD(P)H dehydrogenase (quinone) activity (-1.89), respiratory electron
transport chain (-1.55), cell adhesion (-1.40) and prefoldin complex (-2.02).

## fig4fgh-chronic-up

> The heatmaps of gene sets up-regulated under chronic acidification hold 82 cellular component,
> 308 biological process and 128 molecular function gene sets, including vesicle, endosome and
> Golgi components, purine biosynthesis and carbohydrate catabolism, ribonucleoprotein and
> ubiquitin-related activities, and oxidoreductase, hydrolase and transporter activities.

Counts per ontology of the chronic-OA sets with NES > 0, under the criterion's readings as amended
after this attempt's question (`questions/allen-2026.yaml`): (a) all of them, (b) those not
down-regulated under acute-OA, the sets the deposited script's Figure 4F to H chunk passes to
simplifyEnrichment. The named families are matched under each reading. simplifyEnrichment's
GO_similarity, which keeps only terms it finds in the GO database, cannot be run here.

In [ ]:
up = C[C.NES > 0]
up_script = up[~up.ID.isin(A.ID[A.NES < 0])]
expected_fgh = {"CC": (82, 8.2), "BP": (308, 30.8), "MF": (128, 12.8)}
fam_f = {"vesicle": r"vesicle", "endosome": r"endosom", "Golgi": r"golgi", "purine biosynthesis": r"purine.*biosynthe",
         "carbohydrate catabolism": r"carbohydrate catabol", "ribonucleoprotein": r"ribonucleoprotein", "ubiquitin": r"ubiquitin",
         "oxidoreductase": r"oxidoreductase", "hydrolase": r"hydrolase", "transporter activity": r"transporter activity"}
for name, t in {"(a) all chronic-OA up-regulated sets": up, "(b) up in chronic-OA and not down in acute-OA": up_script}.items():
    n = t.groupby("ONTOLOGY").size()
    within = {o: bool(abs(n.get(o, 0) - v) <= tol) for o, (v, tol) in expected_fgh.items()}
    fams = family_table(t, fam_f, +1)
    print(f"{name}: {n.to_dict()} (paper CC 82, BP 308, MF 128); within tolerance {within}; families with a term {int((fams.terms > 0).sum())} of {len(fams)}")
    print(fams.to_string(index=False), "\n")


**Result: reproduced.** Under reading (b), the chronic-OA up-regulated sets not down-regulated in
acute-OA, the counts are 82 CC, 314 BP and 128 MF, all within tolerance of 82, 308 and 128, and
all 10 named families have at least one up-regulated term. Under reading (a), all chronic-OA
up-regulated sets, the counts are 101 CC, 321 BP and 133 MF, with CC out of tolerance (82 ± 8.2),
and the 10 families hold as well. The claims file added reading (b) in answer to this attempt's
question; under reading (a) alone the claim was partially reproduced.

## chronic-nachr-subunits-down

> Chronic acidification down-regulated transcripts of several nicotinic acetylcholine receptor
> subunits (alpha-7, alpha-10 and beta-4) in the optic lobes.

Transcripts are identified by the protein names of the authors' annotation table (`DB$name`, the
table the script passes to AnnotationForge). Reading (a): core enrichment genes of a chronic-OA
down-regulated synaptic set (fig4e); reading (b): negative shrunken log2 fold change, chronic-OA
against control (`resNormal$CvsCT`).

In [ ]:
names_tab = sq.df_from(ws["DB"]["name"])
nachr = names_tab[names_tab.PNAME.str.contains(r"acetylcholine receptor", case=False, na=False)]
print("Annotated transcripts naming an acetylcholine receptor:", len(nachr))
sub = {"alpha-7": r"alpha[- ]?7(?!\d)", "alpha-10": r"alpha[- ]?10(?!\d)", "beta-4": r"beta[- ]?4(?!\d)"}
syn = C[(C.NES < 0) & C.Description.str.contains(r"synap|transmitter|neurotransmitter", case=False)]
core = set("/".join(syn.core_enrichment).split("/"))
print("Chronic-OA down synaptic sets:", list(syn.Description), "; core enrichment genes:", len(core))
rC = res["CvsCT"]
for s_name, pat in sub.items():
    tx = nachr[nachr.PNAME.str.contains(r"nicotinic|neuronal", case=False) & nachr.PNAME.str.contains(pat, case=False)]
    tx = tx[~tx.PNAME.str.contains("like protein|associated", case=False)]
    rows = rC.reindex(tx.GID)
    print(f"\n{s_name}: {len(tx)} transcripts; in core enrichment {int(tx.GID.isin(core).sum())}; with a fold change {int(rows.log2FoldChange.notna().sum())}, "
          f"negative {int((rows.log2FoldChange < 0).sum())}; smallest adjusted p {rows.padj.min():.4f}")
    if len(tx):
        print(pd.concat([tx.set_index("GID").PNAME, rows[["baseMean", "log2FoldChange", "padj"]]], axis=1).round(4).to_string())

**Result: reproduced.** Under reading (a), each subunit has transcripts among the core enrichment
genes of the chronic-OA down-regulated synaptic sets (alpha-7 2 of 4 transcripts, alpha-10 10 of
29, beta-4 1 of 1). Under reading (b) the tested alpha-7 and beta-4 transcripts have negative
shrunken log2 fold changes (-0.64, -1.03; -0.87) and most alpha-10 transcripts do (14 of 17
tested; 3 positive); none has an adjusted p below 0.09.

## optic-lobe-no-ion-regulation

> No change was seen in optic lobe transcripts associated with acid-base or ion regulatory
> processes.

GO terms whose names contain "pH", "acid-base", "bicarbonate", "proton", "chloride" or "ion
transmembrane transport" (the criterion), significant in either treatment.

In [ ]:
pat = r"\bpH\b|acid-base|bicarbonate|proton|chloride|ion transmembrane transport"
for k, t in gres.items():
    hit = t[t.Description.str.contains(pat, case=True, regex=True) | t.Description.str.contains(r"acid-base|bicarbonate|proton|chloride|ion transmembrane transport", case=False)]
    print(f"{k}: {len(hit)} significant terms")
    if len(hit):
        print(hit[["ONTOLOGY", "Description", "NES", "p.adjust"]].round(4).to_string(index=False))

**Result: not reproduced.** No acute-OA term matches, but three chronic-OA terms do, all
up-regulated: transition metal ion transmembrane transporter activity (NES 2.13, adjusted p =
0.004), organic anion transmembrane transporter activity (1.87, p = 0.033; it matches "ion
transmembrane transport" inside "anion") and metal cation:proton antiporter activity (1.70, p =
0.034). Among the chronic-OA down-regulated sets are also several ion channel activity terms (cell
of fig4e), which the criterion's words do not match.

The criterion's words also catch "organic anion transmembrane transporter activity" (through
"anion") and "transition metal ion transmembrane transporter activity", which are arguably not
acid-base regulation; a question has been filed (`questions/allen-2026.yaml`). Metal cation:proton
antiporter activity matches "proton" and is significant, so the outcome is the same without the
other two.

## Summary

| Claim | Outcome |
|---|---|
| fig1b-metabolic-rate | reproduced (26 squid deposited, 29 in the paper's ANOVA) |
| fig1c-ammonia-excretion | reproduced (the paper's F and p exactly) |
| fig1d-blood-ph | reproduced |
| fig1e-blood-pco2 | reproduced (values a tenth of the deposit's own Eq. 1) |
| fig1f-blood-bicarbonate | reproduced |
| methods-body-size | blocked, data missing |
| stable5-seawater-chemistry | blocked, data missing |
| sfig2-distance-unchanged | reproduced (the paper's H and p exactly) |
| fig2c-velocity-unchanged | reproduced |
| fig2d-acute-upper-zone | reproduced (paper's set found by leaving out one acute-OA squid) |
| stable3-acute-shallower | blocked, data missing |
| fig2e-left-right-unchanged | reproduced |
| fig2f-front-back-unchanged | reproduced |
| fig2g-attack-willingness | reproduced on the counts of the workspace's Figure 2G table (the per-squid file lacks 11 chronic-OA squid) |
| fig2h-hunt-duration | reproduced |
| fig2i-strike-attempts | reproduced |
| sfig3-prey-velocity-unchanged | partially reproduced (CO2 20 min column is a copy of control; time and interaction untestable) |
| fig3c-ratio-declines | not reproduced (criterion's "every frequency above 40 Hz") |
| sfig4a-amplitude-attenuation | blocked, data missing |
| fig3d-cff-unchanged | reproduced |
| fig3d-cff-variance | reproduced |
| rnaseq-assembly | partially reproduced (mapping rate not deposited) |
| fig4a-pca | partially reproduced (rests on reading "spread" as the range; under the SD it would be reproduced) |
| sfig5b-acute-clusters-with-control | reproduced |
| fig4b-shared-down-sets | reproduced (36 under the Venn intersect reading added in answer to a question) |
| fig4c-acute-ridge | reproduced |
| fig4d-acute-tree | reproduced |
| fig4e-chronic-synaptic-down | reproduced |
| fig4fgh-chronic-up | reproduced (82, 314, 128 under the reading added in answer to a question; 101 CC under the other) |
| chronic-nachr-subunits-down | reproduced |
| optic-lobe-no-ion-regulation | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The deposit holds derived values per squid, so the
rates, distances, zone percentages and CFF are taken as deposited, not recomputed from raw
recordings. The transcriptomic claims are tested on the authors' count matrix and their saved
DESeq2, sva and GSEA objects, not from the reads. GSEA and DESeq2 are not re-run, so the outcomes
of fig4b to optic-lobe-no-ion-regulation show that the paper agrees with the authors' saved
results, not an independent recomputation; the DESeq2 fits are checked to rest on the deposited
counts (same counts, same size factors). Welch's ANOVA and Dunnett's T3
are implemented here; the Welch W for pH, PCO2 and [HCO3-] and the Dunnett's T3 intervals differ from
those Supplementary Table 2 prints on the same values, without changing which pairs differ; the
paper's own pH line (W = 5.983 on 2 and 9.196 df, p = 0.00965) is inconsistent, as that W gives
p = 0.022. The left-out acute-OA squid of the zone analysis is found by matching the paper's sums
of squares; in the workspace's figure tables it is the one acute-OA zone value with no distance
beside it, the squid the distance file lacks if the rows of those tables are the same squid in the
same order, which is assumed.

What was found about the dataset. The RSEM count matrix names 16 samples in its header but each
row holds 15 values: the last optic lobe sample has none, and read as the authors' script reads it
(read.csv with row.names = 1) every name shifts by one, so the first value column, by its header
position the gill sample, is labelled as the first control optic lobe. The CO2 10 to 20 min column of the prey shrimp table is a copy
of the control 10 to 20 min column. The prey capture file lacks 11 chronic-OA squid that the
workspace's Figure 2G table and the paper count. The hemolymph PCO2 values are a tenth of what the
formula printed beside them gives, which is the value consistent with the measured TCO2 and pH
under the stated αCO2. The zone file holds an acute-OA squid the paper's analysis
left out, unmarked. The metabolic rate file has 26 squid against the paper's 29. The 3-D tracks,
ERG recordings, seawater chemistry, body mass and mantle length, which the paper's Data
availability statement places in the deposit, are not in it.

In [ ]:
import resource
print(f"Notebook time {time.time() - started:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")